In [1]:
import pandas as pd
import os
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import geopandas as gpd
import numpy as np

%matplotlib inline
pd.set_option('display.max_columns', None)

# Datos electorales

In [2]:
"""data_path = "../data/raw/resultados_electorales/congreso_por_municipio/"
df = None
for file in tqdm(glob.glob(data_path + "*.xlsx")):
    df_single = pd.read_excel(file, header=1, skiprows=4)
    year = os.path.splitext(file)[0].split("/")[-1][3:7]
    month = os.path.splitext(file)[0].split("/")[-1][7:9]
    df_single["Año"] = int(year)
    df_single["Mes"] = int(month)
    df_single["añomes"] = year+month
    print("yearmonth: ", year+month)
    if df is None:
        df = df_single
    else:
        df = pd.concat([df, df_single], ignore_index=True, sort=False)
    break"""

'data_path = "../data/raw/resultados_electorales/congreso_por_municipio/"\ndf = None\nfor file in tqdm(glob.glob(data_path + "*.xlsx")):\n    df_single = pd.read_excel(file, header=1, skiprows=4)\n    year = os.path.splitext(file)[0].split("/")[-1][3:7]\n    month = os.path.splitext(file)[0].split("/")[-1][7:9]\n    df_single["Año"] = int(year)\n    df_single["Mes"] = int(month)\n    df_single["añomes"] = year+month\n    print("yearmonth: ", year+month)\n    if df is None:\n        df = df_single\n    else:\n        df = pd.concat([df, df_single], ignore_index=True, sort=False)\n    break'

In [5]:
data_path = "../../data/raw/resultados_electorales/congreso_por_municipio/02_197706_1.xlsx"
df = pd.read_excel(data_path, header=1, skiprows=4)
year = os.path.splitext(data_path)[0].split("/")[-1][3:7]
month = os.path.splitext(data_path)[0].split("/")[-1][7:9]
df["Año"] = int(year)
df["Mes"] = int(month)
df["añomes"] = year+month
print("yearmonth: ", year+month)


yearmonth:  197706


In [6]:
df.columns

Index(['Nombre de Comunidad', 'Código de Provincia', 'Nombre de Provincia',
       'Código de Municipio', 'Nombre de Municipio', 'Población',
       'Número de mesas', 'Total censo electoral', 'Total votantes',
       'Votos válidos', 'Votos a candidaturas', 'Votos en blanco',
       'Votos nulos', 'UCD', 'PSOE', 'PCE', 'AP', 'PSP-US', 'PDPC', 'PNV',
       'FDC-EDC', 'UDC-IDCC', 'EC-FED', 'FDI', 'ASDCI', 'AET', 'AN18', 'RSE',
       'EE', 'FJONSA', 'FUT', 'CAIC', 'ESB', 'PSPV', 'INDEP', 'DCV', 'PSG',
       'FJONS', 'UNAI', 'BNPG', 'AFN', 'INDEP.1', 'PSOE-H', 'URA', 'LLIGA',
       'UAN', 'ANEPA-CP', 'PCU', 'FL', 'DIV', 'CJA', 'CUPS', 'CCIA', 'UAB',
       'URAS', 'FNI', 'PPCAN', 'MS', 'ASA', 'DSCC', 'MFA', 'ANV', 'INDEP.2',
       'INDEP.3', 'INDEP.4', 'DCAR', 'UDIB', 'INDEP.5', 'FN', 'CUP', 'PSCAN',
       'FAA', 'PIM', 'INDEP.6', 'PRSV', 'ICU', 'PPROV', 'PSDE', 'INDEP.7',
       'INDEP.8', 'PDG', 'INDEP.9', 'INDEP.10', 'INDEP.11', 'AEICYU', 'CUIR',
       'CIPYE', 'ADEC', 'PLI', 'A

In [7]:
# limpiar columnas index y rellenar NA
df = df[df.columns.drop(list(df.filter(regex='Unnamed')))]
df = df.fillna(0)
# eliminar columnas con menso de x votos
#df_sum = pd.DataFrame(df.sum(axis = 0, skipna = True, numeric_only=True))
#df_sum[df_sum[0] == 0].index

In [8]:
# crear codigo ine
df["CODIGOINE"] = df["Código de Provincia"].astype(int).astype(str).str.zfill(2) + df["Código de Municipio"].astype(int).astype(str).str.zfill(3)

In [9]:
df.tail(5)

,Nombre de Comunidad,Código de Provincia,Nombre de Provincia,Código de Municipio,Nombre de Municipio,Población,Número de mesas,Total censo electoral,Total votantes,Votos válidos,Votos a candidaturas,Votos en blanco,Votos nulos,UCD,PSOE,PCE,AP,PSP-US,PDPC,PNV,FDC-EDC,UDC-IDCC,EC-FED,FDI,ASDCI,AET,AN18,RSE,EE,FJONSA,FUT,CAIC,ESB,PSPV,INDEP,DCV,PSG,FJONS,UNAI,BNPG,AFN,INDEP.1,PSOE-H,URA,LLIGA,UAN,ANEPA-CP,PCU,FL,DIV,CJA,CUPS,CCIA,UAB,URAS,FNI,PPCAN,MS,ASA,DSCC,MFA,ANV,INDEP.2,INDEP.3,INDEP.4,DCAR,UDIB,INDEP.5,FN,CUP,PSCAN,FAA,PIM,INDEP.6,PRSV,ICU,PPROV,PSDE,INDEP.7,INDEP.8,PDG,INDEP.9,INDEP.10,INDEP.11,AEICYU,CUIR,CIPYE,ADEC,PLI,ADC,PAE,FEI,BAI,INDEP.12,CAI,Año,Mes,añomes,CODIGOINE
8109,Comunitat Valenciana,46,Valencia / València,261,YATOVA ...,2058,2,1311,1104,1103,1103,0,1,399,403,165,85,24,0,0,6,0,0,5,1,1,1,3,0,2,3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1977,6,197706,46261
8110,Comunitat Valenciana,46,Valencia / València,262,YESA LA ...,518,1,407,328,327,326,1,1,219,33,11,10,11,0,0,2,0,0,5,0,0,2,4,0,0,1,0,0,19,0,0,0,0,0,0,0,0,0,0,0,0,8,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1977,6,197706,46262
8111,Comunitat Valenciana,46,Valencia / València,263,ZARRA ...,602,1,427,355,353,352,1,2,194,70,9,5,5,0,0,0,0,0,3,9,0,12,2,0,8,5,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1977,6,197706,46263
8112,Ciudad de Ceuta,51,Ceuta,1,CEUTA ...,67077,57,31672,24573,24285,24218,67,288,8804,7886,0,2914,2763,0,0,0,0,0,0,0,0,0,752,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1099,0,0,0,0,0,0,0,1977,6,197706,51001
8113,Ciudad de Melilla,52,Melilla,1,MELILLA ...,60191,44,25359,19268,19048,18949,99,220,10723,5186,966,2074,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1977,6,197706,52001


In [10]:
df.shape

(8114, 99)

# Mapa España por munipios

In [11]:
#map_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.geojson"
#guardar
#geodf_reduced.to_file(geojson_path, driver = "GeoJSON")

In [15]:
geojson_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN_reduced.geojson"

In [16]:
#cargar
with open(geojson_path) as geofile:
    geojson = json.load(geofile)

# Mapa de ganadores

In [17]:
key_columns = ['Nombre de Comunidad', 'Código de Provincia', 'Nombre de Provincia',
       'Código de Municipio', 'Nombre de Municipio', 'Población',
       'Número de mesas', 'Total censo electoral', 'Total votantes',
       'Votos válidos', 'Votos a candidaturas', 'Votos en blanco',
       'Votos nulos','Año', 'Mes', 'añomes', 'CODIGOINE']
party_columns = [x for x in df.columns if x not in key_columns]

In [18]:
df["ganador"] = df[party_columns].idxmax(axis=1)

In [19]:
df["ganador"].unique()

array(['UCD', 'PSOE', 'AP', 'PCE', 'FDI', 'PSP-US', 'INDEP.4', 'CAIC',
       'FDC-EDC', 'DCAR', 'AN18', 'CCIA', 'FJONS', 'AEICYU', 'INDEP.6',
       'FJONSA', 'UDC-IDCC', 'PDPC', 'LLIGA', 'EC-FED', 'PSG', 'UNAI',
       'UAN', 'AFN', 'MFA', 'FNI', 'AET', 'INDEP.1', 'PNV', 'EE',
       'INDEP.9', 'INDEP.5', 'INDEP', 'FL'], dtype=object)

# Gráfico

In [20]:
color_map = {'PSOE':"#EF553B", 
             'PP':"#636EFA", 
             'IU':"#2CA02C", 
             'CHA':"#B82E2E", 
             'UM':"#EECA3B", 
             'CC':"#17BECF", 
             'CIU':"#3366CC", 
             'ERC':"#FF9900", 
             'BNG':"#0DF9FF", 
             'HB':"#222A2A",
             'CDN':"#AF0038", 
             'EA':"#66AA00", 
             'EAJ-PNV':"#479B55", 
             'P.RIOJANO':"#86CE00"}

In [21]:
df_filtered = df[["ganador", "CODIGOINE"]]

In [22]:
df_filtered.shape

(8114, 2)

In [23]:
#colores, hover

In [24]:
"""fig = px.choropleth_mapbox(data_frame=df_filtered, geojson=geojson, locations='CODIGOINE', featureidkey="properties.CODIGOINE", 
                           color='ganador', mapbox_style="carto-positron", color_discrete_map=color_map,
                           center={"lat": 40.463667, "lon": -3.74922}, scope="europe", zoom=4)
fig.update_traces(marker_line_width=0)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_geos(showcountries=False, showcoastlines=False, showland=False, fitbounds="locations")
fig.show()"""

'fig = px.choropleth_mapbox(data_frame=df_filtered, geojson=geojson, locations=\'CODIGOINE\', featureidkey="properties.CODIGOINE", \n                           color=\'ganador\', mapbox_style="carto-positron", color_discrete_map=color_map,\n                           center={"lat": 40.463667, "lon": -3.74922}, scope="europe", zoom=4)\nfig.update_traces(marker_line_width=0)\nfig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})\nfig.update_geos(showcountries=False, showcoastlines=False, showland=False, fitbounds="locations")\nfig.show()'